# shrimp365 비전 — 흰다리새우 계수 모델 학습

구글 드라이브 `00. 데이터/01. 흰다리새우 학습 데이터` 의 라벨 데이터로 YOLOv8 검출 모델을
학습하고, 라즈베리파이 4 에 올릴 ONNX 까지 만든다.

**왜 Colab 인가.** 학습 데이터가 5천 장이라 드라이브에서 다른 곳으로 옮기는 것 자체가 부담이다.
Colab 은 드라이브를 직접 마운트하고 무료 GPU 를 쓸 수 있어, 데이터를 옮기지 않고 학습할 수 있다.

**목표 지표** (수행계획서 검수 기준)

| 항목 | 목표 |
|---|---|
| 계수 오차율 (일반) | ±20% 이내 |
| 계수 오차율 (탁도·겹침) | ±30% 이내 |
| 장당 처리 시간 | 1~3초 (라즈베리파이 기준) |
| 참고 지표 mAP@0.5 | 0.85 이상 |

mAP 는 참고용이다. **합격을 결정하는 것은 계수 오차율**이므로 5단계를 반드시 돌린다.

---

### 시작 전 한 가지

상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 로 바꾸고 저장한다.
CPU 로 돌리면 100 에포크에 수십 시간이 걸린다.

In [ ]:
# 0단계 — GPU 확인
# GPU 가 안 보이면 런타임 유형을 T4 GPU 로 바꾸세요.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'GPU 없음'

## 1단계 — 드라이브 마운트

팝업이 뜨면 계정을 고르고 접근을 허용한다. 학습 데이터는 읽기만 하고, 결과 파일만 쓴다.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/00. 데이터/01. 흰다리새우 학습 데이터')
print('학습 데이터 폴더:', BASE)
print('상태:', '있음' if BASE.is_dir() else '없음 — 경로를 확인하세요')
if BASE.is_dir():
    for child in sorted(BASE.iterdir()):
        print('  -', child.name)

## 2단계 — 학습 도구 가져오기

저장소 `vision/ai/trainer/` 의 스크립트 4개를 쓴다. 저장소가 비공개라 Colab 에서 바로 clone 할 수
없으므로, **한 번만** 파일을 올려 드라이브에 넣어 두면 다음 실행부터는 자동으로 재사용한다.

올릴 파일 (GitHub 웹에서 각 파일 → `Download raw file`)

- `prepare_dataset.py`
- `train.py`
- `eval_count.py`
- `export_edge.py`

In [ ]:
import shutil
from pathlib import Path

TRAINER = Path('/content/trainer')
TRAINER.mkdir(exist_ok=True)
CACHE = Path('/content/drive/MyDrive/shrimp365-trainer')  # 한 번 올려두면 계속 쓴다
NEEDED = ['prepare_dataset.py', 'train.py', 'eval_count.py', 'export_edge.py']

missing = [name for name in NEEDED if not (CACHE / name).exists()]
if missing:
    print('드라이브 캐시에 없는 파일:', missing)
    print('아래 버튼으로 해당 .py 파일들을 올리세요(한 번만 하면 됩니다).')
    from google.colab import files

    uploaded = files.upload()
    CACHE.mkdir(parents=True, exist_ok=True)
    for name in uploaded:
        shutil.copy(name, CACHE / name)

for name in NEEDED:
    shutil.copy(CACHE / name, TRAINER / name)
print('준비된 도구:', sorted(path.name for path in TRAINER.glob('*.py')))

In [ ]:
# 아래 단계에서 쓰는 실행 헬퍼.
# `!명령` 대신 이것을 쓰는 이유: 한글·공백이 들어간 드라이브 경로를 안전하게 인용하고,
# 명령이 실패하면 조용히 넘어가지 않고 여기서 멈춘다(실패를 못 본 채 다음 단계로
# 넘어가면 원인을 찾기가 훨씬 어려워진다).
import shlex
import subprocess


def run(*argv):
    cmd = ' '.join(shlex.quote(str(a)) for a in argv)
    print('$', cmd, flush=True)
    code = subprocess.call(cmd, shell=True)
    if code != 0:
        raise RuntimeError(f'명령이 실패했습니다(종료 코드 {code}): {cmd}')

## 3단계 — 데이터셋 준비 (여기가 핵심)

드라이브의 두 데이터셋을 그대로 학습에 넣으면 안 되는 이유가 있다.

1. **`shrimp_cf` 의 라벨은 바운딩 박스가 아니라 세그멘테이션 폴리곤**이다. 한 줄이
   `0 x1 y1 x2 y2 … x24 y24`(49개 필드)라서 detect 모델 학습에 그대로 넣으면 좌표가
   잘못 해석된다. → 폴리곤의 최소/최대 좌표로 박스를 만든다.
2. **클래스 이름이 다르다.** `shrimp_cf` 는 `shrimp`, `Counting PL` 은 `Shrimp-baby`(자어)인데
   둘 다 nc=1 이라 인덱스가 0 으로 겹친다. → 기본값은 단일 클래스(`shrimp`)로 병합한다.
   운영 검출기는 클래스를 구분하지 않고 박스를 세므로 이게 실제와 맞는다. 성체/자어를
   나누고 싶으면 `--stage-classes` 를 붙인다.
3. **분할 누수 검사.** 같은 원본 프레임이 train 과 valid 에 함께 있으면 검증 점수가 부풀려져
   실제보다 잘하는 것처럼 보인다. 파일 해시와 Roboflow 원본 파일명으로 검사하고, 발견되면
   train 쪽에서 뺀다(`--on-leakage drop`).

In [ ]:
# 두 Roboflow 내보내기 폴더를 자동으로 찾는다(data.yaml + train 폴더가 있는 곳).
found = []
for path in sorted(BASE.glob('*/*')):
    if (path / 'data.yaml').exists() and (path / 'train').is_dir():
        found.append(path)
        print('발견:', path)
assert found, 'YOLOv8 내보내기 폴더를 찾지 못했습니다. 압축을 풀어 두었는지 확인하세요.'

In [ ]:
import re


def source_name(path):
    """폴더 이름을 영문·숫자·밑줄만 남긴 원본 이름으로 바꾼다."""
    stem = path.name.split('.')[0]
    return re.sub(r'[^0-9A-Za-z]+', '_', stem).strip('_').lower() or 'src'


DATASET = '/content/datasets/shrimp'
argv = ['python', '/content/trainer/prepare_dataset.py']
for path in found:
    argv += ['--source', f'{source_name(path)}={path}']
argv += ['--out', DATASET, '--on-leakage', 'drop',
         '--report', '/content/prepare_report.json']
run(*argv)

### 변환 결과를 눈으로 확인

폴리곤→박스 변환이 틀리면 학습은 조용히 망가진다(오류 없이 성능만 안 나온다).
무작위 몇 장에 라벨 박스를 그려 새우 위에 제대로 올라가는지 본다.

In [ ]:
import random

import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

images = sorted(Path(DATASET + '/train/images').iterdir())
random.seed(0)
picks = random.sample(images, min(6, len(images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, image_path in zip(axes.ravel(), picks, strict=False):
    image = Image.open(image_path)
    width, height = image.size
    ax.imshow(image)
    label_path = Path(str(image_path).replace('/images/', '/labels/')).with_suffix('.txt')
    count = 0
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        _, xc, yc, w, h = (float(v) for v in line.split())
        ax.add_patch(patches.Rectangle(
            ((xc - w / 2) * width, (yc - h / 2) * height),
            w * width, h * height,
            fill=False, edgecolor='lime', linewidth=1.2))
        count += 1
    ax.set_title(f'{image_path.name[:28]} — {count}마리', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4단계 — 학습

`yolov8n`(가장 작은 모델)으로 시작한다. 파이에서 돌려야 하므로 모델을 키우는 건 마지막 수단이다.
T4 GPU 기준 5천 장 × 100 에포크는 대략 2~4시간이다. 세션이 끊길 수 있으니 결과가
`/content/runs` 에 쌓이는 것을 확인하고, 시간이 부담되면 먼저 `--epochs 50` 으로 돌려 본다.

In [ ]:
!pip install -q ultralytics

import ultralytics

ultralytics.checks()

In [ ]:
run('python', '/content/trainer/train.py',
    '--data', f'{DATASET}/data.yaml',
    '--base-model', 'yolov8n.pt',
    '--epochs', 100, '--imgsz', 640, '--batch', 16,
    '--project', '/content/runs', '--name', 'shrimp')

In [ ]:
import os

from IPython.display import Image as IPyImage
from IPython.display import display

BEST = '/content/runs/shrimp/weights/best.pt'
print('학습 결과:', BEST, '→', '있음' if os.path.exists(BEST) else '없음(위 로그 확인)')
for name in ('results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg'):
    path = f'/content/runs/shrimp/{name}'
    if os.path.exists(path):
        print(name)
        display(IPyImage(filename=path, width=720))

## 5단계 — 계수 오차율 검증 (합격 판정)

mAP 가 높아도 마리 수가 틀리면 이 사업에서는 실패다. 장당 정답 개체 수와 검출 개수를 직접
비교해 오차율을 낸다. 한 번의 추론 결과로 여러 신뢰도 임계값을 다시 세어 **최적 conf** 도
함께 추천하니, 출력된 추천값을 운영 설정(`CONFIDENCE_THRESHOLD`)에 반영한다.

탁하거나 겹침이 심한 이미지 파일명을 한 줄에 하나씩 적은 목록을 `--hard-list` 로 주면
그 묶음만 ±30% 기준으로 따로 판정한다.

In [ ]:
run('python', '/content/trainer/eval_count.py',
    '--weights', BEST, '--data', f'{DATASET}/data.yaml',
    '--split', 'valid', '--conf', 0.25,
    '--json', '/content/count_eval_valid.json')

In [ ]:
# test 분할이 있으면 학습·검증에 한 번도 쓰이지 않은 데이터로 최종 확인한다.
test_dir = DATASET + '/test/images'
if os.path.isdir(test_dir) and os.listdir(test_dir):
    run('python', '/content/trainer/eval_count.py',
        '--weights', BEST, '--data', f'{DATASET}/data.yaml',
        '--split', 'test', '--conf', 0.25,
        '--json', '/content/count_eval_test.json')
else:
    print('test 분할이 없습니다 — valid 결과로 판단합니다.')

## 6단계 — 라즈베리파이용 ONNX 내보내기

파이 4 는 GPU 가 없다. `best.pt` 를 그대로 올리면 torch 설치가 무겁고 640 해상도에서 한 장에
2초 안팎이 걸린다. ONNX 로 내보내면 파이에서 `onnxruntime` 만 설치해 돌릴 수 있고, 해상도를
512/416 으로 낮추면 처리 시간이 크게 줄어든다.

두 해상도를 모두 만들어 두고 파이에서 속도와 정확도를 비교해 고른다.
**해상도를 바꾸면 정확도도 바뀌므로, 내보낸 모델로 계수 오차율을 다시 재야 한다.**

In [ ]:
for imgsz in (512, 416):
    run('python', '/content/trainer/export_edge.py',
        '--weights', BEST, '--format', 'onnx', '--imgsz', imgsz)
    # 항상 best.onnx 로 나오므로 해상도를 파일명에 남긴다.
    run('mv', '/content/runs/shrimp/weights/best.onnx',
        f'/content/runs/shrimp/weights/best_{imgsz}.onnx')
run('ls', '-la', '/content/runs/shrimp/weights/')

In [ ]:
# 내보낸 ONNX 로 계수 오차율을 다시 확인한다.
# 해상도를 낮추면 작은 개체를 놓칠 수 있어, 파이에 올릴 모델로 직접 재야 한다.
for imgsz in (512, 416):
    path = f'/content/runs/shrimp/weights/best_{imgsz}.onnx'
    if not os.path.exists(path):
        continue
    print(f'=== imgsz {imgsz} ===')
    run('python', '/content/trainer/eval_count.py',
        '--weights', path, '--data', f'{DATASET}/data.yaml',
        '--split', 'valid', '--conf', 0.25,
        '--json', f'/content/count_eval_onnx_{imgsz}.json')

## 7단계 — 결과를 드라이브에 저장

Colab 런타임은 끊기면 사라진다. 가중치와 평가 결과를 드라이브로 옮겨 둔다.

In [ ]:
import datetime
import glob

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
OUT_DIR = Path(f'/content/drive/MyDrive/shrimp365-models/{stamp}')
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATTERNS = (
    '/content/runs/shrimp/weights/best.pt',
    '/content/runs/shrimp/weights/*.onnx',
    '/content/runs/shrimp/results.csv',
    '/content/runs/shrimp/results.png',
    '/content/count_eval_*.json',
    '/content/prepare_report.json',
)
for pattern in PATTERNS:
    for path in glob.glob(pattern):
        shutil.copy(path, OUT_DIR / os.path.basename(path))

print('저장 위치:', OUT_DIR)
for path in sorted(OUT_DIR.iterdir()):
    print(f'  {path.name}  ({path.stat().st_size / 1e6:.1f} MB)')

## 다음 단계 — 사람이 할 일

1. 드라이브 `shrimp365-models/<날짜>/` 에서 `best_512.onnx`(또는 `best.pt`)를 내려받는다.
2. 파이에 올린다.

   ```bash
   scp best_512.onnx pi@<파이주소>:/tmp/shrimp_yolov8n.onnx
   ssh pi@<파이주소> 'sudo mv /tmp/shrimp_yolov8n.onnx /opt/shrimp365-vision/ai/models/'
   ```
3. 파이에서 실제 처리 시간을 잰다(여기서 나온 숫자가 검수 근거다).

   ```bash
   cd /opt/shrimp365-vision
   sudo .venv/bin/pip install -e '.[edge]'
   .venv/bin/python ai/trainer/export_edge.py \
       --weights ai/models/shrimp_yolov8n.onnx --bench --runs 30
   ```
4. `/etc/shrimp365-vision/env` 를 고치고 서비스를 재시작한다.

   ```bash
   MODEL_PATH=./ai/models/shrimp_yolov8n.onnx
   CONFIDENCE_THRESHOLD=<5단계에서 추천된 값>
   ```

   ```bash
   sudo systemctl restart shrimp365-vision
   ```
5. 학습에 쓴 설정과 평가 결과를 저장소에 남긴다(가중치 파일 자체는 커밋하지 않는다).

자세한 절차는 저장소 `docs/VISION_MODEL_TRAINING.md` 를 본다.